# 🇮🇳 Indian Currency Recognition Pipeline for Visually Impaired Navigation
=====================================================================
- **Target Hardware**: NVIDIA GeForce RTX 4070 (12GB VRAM, CUDA 12.6, AMP)
- **Frameworks**: PyTorch 2.x, Torchvision, Scikit-Learn, ONNX Runtime
- **Denominations**: ₹10, ₹20, ₹50, ₹100, ₹200, ₹500, ₹2000
- **Dataset Splitting**: 70% Train / 15% Val / 15% Test (Stratified)


## 1. ⚙️ Deterministic Setup, Hardware & Environment Configuration
Set deterministic random seeds across PyTorch, NumPy, and CUDA, and initialize execution device with automatic mixed precision (AMP) capabilities for RTX 4070.

In [ ]:
import os
import sys
import time
import json
import random
import shutil
import warnings
from pathlib import Path

# Ensure UTF-8 output on Windows console
if sys.platform.startswith("win"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, models, transforms
from torch.amp import GradScaler, autocast

import sklearn.metrics as metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import onnx

warnings.filterwarnings("ignore")

# 1. Deterministic Random Seeds & Device Setup
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 70)
print(f"🔥 [PyTorch Version]    : {torch.__version__}")
print(f"⚡ [Execution Device]   : {device}")
if torch.cuda.is_available():
    print(f"🎮 [GPU Model]          : {torch.cuda.get_device_name(0)}")
    print(f"💾 [Total VRAM]         : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"✨ [CUDA Runtime]       : {torch.version.cuda}")
    print(f"⚡ [AMP (FP16) Ready]   : Supported & Enabled for Tensor Cores")
print("=" * 70)


## 2. 🏷️ Numerical Class Sorting & Dataset Discovery
Standard `ImageFolder` sorts folder names alphabetically (`100` before `20`), which causes label mismatch. We use a custom `NumericalImageFolder` to ensure strictly ordered numerical classes (`10`, `20`, `50`, `100`, `200`, `500`, `2000`) matching label indices `0..6`.

In [ ]:
DATASET_ROOT = Path("Indian Currency Dataset")
SAVE_DIR = Path("currency_models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset directory '{DATASET_ROOT}' not found!")

# Custom ImageFolder with strict numerical class ordering
class NumericalImageFolder(datasets.ImageFolder):
    def find_classes(self, dir):
        classes, _ = super().find_classes(dir)
        sorted_classes = sorted(classes, key=lambda x: int(x) if x.isdigit() else x)
        sorted_class_to_idx = {c: i for i, c in enumerate(sorted_classes)}
        return sorted_classes, sorted_class_to_idx

raw_dataset = NumericalImageFolder(root=str(DATASET_ROOT))
CLASS_NAMES = raw_dataset.classes
CLASS_TO_IDX = raw_dataset.class_to_idx
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# Save canonical label mapping to json
with open(SAVE_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump({
        "class_names": CLASS_NAMES,
        "class_to_idx": CLASS_TO_IDX,
        "idx_to_class": IDX_TO_CLASS
    }, f, indent=4)

print(f"📦 [Classes Detected]   : {CLASS_NAMES}")
print(f"🏷️ [Class Mapping]      : {CLASS_TO_IDX}")
print(f"📁 [Total Raw Images]   : {len(raw_dataset)}")


## 3. 🔄 Assistive Vision Augmentation & Stratified Splitting
Using **Stratified Splitting** ensures identical class distribution across Train (70%), Validation (15%), and Test (15%) partitions. Transforms include perspective, orientation, and lighting variations.

In [ ]:
# 3. Assistive Vision Data Augmentation Pipelines
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]
IMG_SIZE  = 224
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), shear=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

class TransformedSubset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __getitem__(self, idx):
        x, y = self.dataset[self.indices[idx]]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.indices)

# Stratified Dataset Partitioning
targets = [sample[1] for sample in raw_dataset.samples]
all_indices = np.arange(len(raw_dataset))

# 70% Train, 30% Temp (15% Val, 15% Test)
train_idx, temp_idx = train_test_split(
    all_indices, test_size=0.30, stratify=targets, random_state=42
)
temp_targets = [targets[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_targets, random_state=42
)

train_dataset = TransformedSubset(raw_dataset, train_idx, transform=train_transforms)
val_dataset   = TransformedSubset(raw_dataset, val_idx,   transform=val_test_transforms)
test_dataset  = TransformedSubset(raw_dataset, test_idx,  transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"📊 [Stratified Splits]  : Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")


## 4. 🧠 Deep Learning Architecture: MobileNetV3-Large Transfer Learning
Build an optimized lightweight classification network with pretrained feature extraction and custom multi-layer classifier head with Hardswish activations and Dropout.

In [ ]:
# 4. Neural Network Architecture (MobileNetV3-Large Transfer Learning)
def build_currency_classifier(num_classes=7):
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.Hardswish(),
        nn.Dropout(p=0.35),
        nn.Linear(512, 128),
        nn.BatchNorm1d(128),
        nn.Hardswish(),
        nn.Dropout(p=0.20),
        nn.Linear(128, num_classes)
    )
    return model

model = build_currency_classifier(num_classes=len(CLASS_NAMES)).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 18
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

print(f"✅ Model Initialized on {device} with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.")


## 5. 🚀 High-Performance GPU Training Pipeline (RTX 4070 AMP)
Train the network with mixed precision FP16 Tensor Core acceleration, Cosine Annealing learning rate schedule, and automated best-checkpoint preservation.

In [ ]:
# 5. Training Loop with GPU Mixed Precision
best_val_acc = 0.0
model_save_path = SAVE_DIR / "best_currency_model.pth"

print("\n" + "=" * 70)
print(f"🚀 Starting Training ({EPOCHS} Epochs on RTX 4070 AMP)...")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # Train Phase
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data).item()
        train_total += labels.size(0)

    train_acc = (train_correct / train_total) * 100.0
    train_epoch_loss = train_loss / train_total

    # Validation Phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with autocast('cuda', enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data).item()
            val_total += labels.size(0)

    val_acc = (val_correct / val_total) * 100.0
    val_epoch_loss = val_loss / val_total

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    duration = time.time() - epoch_start
    saved_flag = ""

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': best_val_acc,
            'class_names': CLASS_NAMES,
            'class_to_idx': CLASS_TO_IDX,
            'idx_to_class': IDX_TO_CLASS
        }, model_save_path)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': best_val_acc,
            'class_names': CLASS_NAMES
        }, "best_currency_model.pth")
        saved_flag = f" ⭐ [BEST: {best_val_acc:.2f}%]"

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({duration:.1f}s) | "
          f"Train Loss: {train_epoch_loss:.4f} - Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f} - Acc: {val_acc:.2f}% | "
          f"LR: {current_lr:.6f}{saved_flag}")

print(f"\n✅ Training Complete! Best Validation Accuracy: {best_val_acc:.2f}%")


## 6. 🧪 In-Depth Held-Out Test Set Evaluation
Evaluate the optimal checkpoint on unseen test samples and calculate overall test accuracy and per-denomination precision metrics.

In [ ]:
# 6. Evaluation on Held-Out Test Set
print("\n" + "=" * 70)
print("🧪 Evaluating on Held-Out Test Set...")
print("=" * 70)

ckpt = torch.load(model_save_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

y_true, y_pred, y_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        with autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_probs.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
test_accuracy = (np.sum(y_true == y_pred) / len(y_true)) * 100.0
print(f"🎯 Test Set Accuracy: {test_accuracy:.2f}%\n")

# Per-Class Accuracy Table
print(f"{'Class (Denomination)':<22} | {'Total Samples':<15} | {'Correct':<10} | {'Accuracy'}")
print("-" * 70)
for idx, cls_name in enumerate(CLASS_NAMES):
    mask = (y_true == idx)
    cls_total = np.sum(mask)
    cls_correct = np.sum(y_pred[mask] == idx)
    cls_acc = (cls_correct / cls_total) * 100.0 if cls_total > 0 else 0.0
    print(f"Rs. {cls_name:<18} | {cls_total:<15} | {cls_correct:<10} | {cls_acc:.2f}%")


## 7. 🔊 Real-Time Assistive Vision & Audio Guidance Simulation
Simulate real-time banknote inspection with spoken feedback prompts designed for visually impaired navigation assistance.

In [ ]:
# 7. 10-Sample Test Verification
print("\n" + "=" * 75)
print("🔊 10-SAMPLE VERIFICATION INFERENCE & AUDIO GUIDANCE:")
print("=" * 75)
print(f"{'#':<3} | {'True Denomination':<18} | {'Predicted':<14} | {'Confidence':<10} | {'Voice Prompt'}")
print("-" * 75)

# Pick 10 random samples from the held-out test set
random.seed(42)
sample_indices = random.sample(range(len(test_dataset)), min(10, len(test_dataset)))
correct_10 = 0

for i, s_idx in enumerate(sample_indices):
    tensor_im, true_label = test_dataset[s_idx]
    true_d = CLASS_NAMES[true_label]
    input_tensor = tensor_im.unsqueeze(0).to(device)

    with torch.no_grad():
        with autocast('cuda', enabled=torch.cuda.is_available()):
            out = model(input_tensor)
            probs = F.softmax(out, dim=1)[0]
            p_idx = torch.argmax(probs).item()
            conf = probs[p_idx].item()
            pred_d = CLASS_NAMES[p_idx]

    is_corr = (pred_d == true_d)
    if is_corr:
        correct_10 += 1
    prompt = f"₹{pred_d} note detected. Clear view." if conf >= 0.90 else f"₹{pred_d} note ({conf*100:.1f}%). Hold still."
    status = "[OK]" if is_corr else "[X]"
    print(f"{i+1:<3} | Rs. {true_d:<14} | Rs. {pred_d:<10} | {conf*100:>6.2f}%    | {status} {prompt}")

print("-" * 75)
print(f"10-Sample Result: {correct_10}/10 ({(correct_10/10)*100:.1f}%)\n")


## 8. ⚡ Mobile & Web Edge Deployment: ONNX Export (IR Version 9)
Export the trained PyTorch weights to an optimized ONNX computational graph with dynamic batch dimensions and explicit IR Version 9 metadata for cross-platform Android and Web execution.

In [ ]:
# 8. Export to ONNX (IR Version 9 for Mobile & Web)
print("=" * 70)
print("⚡ Exporting ONNX Model for Android, Web & Hugging Face...")
print("=" * 70)

onnx_path = str(SAVE_DIR / "currency_model.onnx")
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['input_image'],
    output_names=['class_logits'],
    dynamic_axes={'input_image': {0: 'batch_size'}, 'class_logits': {0: 'batch_size'}},
    opset_version=14,
    dynamo=False
)

# Set IR Version 9
onnx_model = onnx.load(onnx_path)
onnx_model.ir_version = 9
onnx.checker.check_model(onnx_model)
onnx.save(onnx_model, onnx_path)
shutil.copy2(onnx_path, "currency_model.onnx")

print(f"✅ ONNX Model saved to: {onnx_path} and currency_model.onnx")
print(f"✨ ONNX IR Version: {onnx_model.ir_version} (Fully compatible with Android & Hugging Face!)")
